In [1]:
import torch
from mini_whisper import *
from mini_whisper.transformer.MHA_simple import SimpleTransformerBlock
from mini_whisper.transformer.MHA import TransformerBlock

In [2]:
SPLIT = "dev-clean"
DATA_DIR = "./data"
FOLDER_IN_ARCHIVE = "LibriSpeech"
BATCH_SIZE = 16
N_MELS = 80
D_MODEL = 128
N_HEADS = 8

print("=" * 60)
print("Mini-Whisper Training - Data Loading & Preprocessing")
print("=" * 60)


Mini-Whisper Training - Data Loading & Preprocessing


In [3]:
print(f"\nCreating DataLoader for: {DATA_DIR}")
print(f"Batch size: {BATCH_SIZE}")

dataloader = LibriSpeechAudioPreprocessingDataLoader(
    split=SPLIT,
    root_dir=DATA_DIR,
    folder_in_archive=FOLDER_IN_ARCHIVE,
    download_dataset=True,
    batch_size=BATCH_SIZE,
    shuffle=True,
    num_workers=2,
    n_mel_bins=N_MELS
)

print(f"\nDataLoader created with {len(dataloader.dataset)} samples")
print(f"  Number of batches: {len(dataloader)}")



Creating DataLoader for: ./data
Batch size: 16

DataLoader created with 2703 samples
  Number of batches: 169


In [4]:
print(f"\nInitializing AudioEncoderStem (n_mels={N_MELS}, d_model={D_MODEL})")
stem = AudioEncoderStem(n_mels=N_MELS, d_model=D_MODEL)
stem.eval()
print("Encoder stem initialized")



Initializing AudioEncoderStem (n_mels=80, d_model=128)
Encoder stem initialized


In [5]:
print(f"\nProcessing first batch...")
batch = next(iter(dataloader))

log_mels = batch['log_mel']  # (B, n_mels, T)
transcripts = batch['transcript']
audio_paths = batch['audio_path']

print(f"\nBatch contents:")
print(f"  Log-mel shape: {log_mels.shape}")
print(f"  Number of transcripts: {len(transcripts)}")



Processing first batch...

Batch contents:
  Log-mel shape: torch.Size([16, 80, 3000])
  Number of transcripts: 16


In [6]:
print(f"\nFirst 3 samples in batch:")
for i in range(min(3, len(transcripts))):
    print(f"  {i+1}. {audio_paths[i]}")
    print(f"     Transcript: {transcripts[i][:60]}...")



First 3 samples in batch:
  1. 3081/166546/3081-166546-0011.flac
     Transcript: THE BOYS LOOK WIDE AWAKE ENOUGH BUT WHO CAN TELL I WOULD SOO...
  2. 3170/137482/3170-137482-0038.flac
     Transcript: NOT VERY LONG I ANSWERED AND I WILL TEACH YOU AS YOU WISH AL...
  3. 6313/76958/6313-76958-0015.flac
     Transcript: STACY GRUMBLED TURNED OVER AND WENT TO SLEEP AGAIN...


In [7]:
# Let's first start just with the stem
with torch.no_grad():
    batch_features = stem(log_mels)

print(f"\nEncoder output shape: {batch_features.shape}")



Encoder output shape: torch.Size([16, 1500, 128])


In [8]:
# Now let us add a layer of the encoder self-attention transformer block
print(f"\nInitializing AudioEncoderLayer (d_model={D_MODEL})")
encoder_layer = SimpleTransformerBlock(d_model=D_MODEL, n_head=N_HEADS)
with torch.no_grad():
    x = stem(log_mels)
    z = encoder_layer(x)
print(f"Output shape after encoder layer: {batch_features.shape}")



Initializing AudioEncoderLayer (d_model=128)
Output shape after encoder layer: torch.Size([16, 1500, 128])


In [9]:
# Okay, time for a encoder+decoder pair
print(f"\nInitializing AudioEncoderLayer (d_model={D_MODEL})")
encoder_layer = SimpleTransformerBlock(d_model=D_MODEL, n_head=N_HEADS)
decoder_layer = TransformerBlock(d_model=D_MODEL, n_head=N_HEADS, cross_attention=True)
with torch.no_grad():
    x = stem(log_mels)
    z = encoder_layer(x)
    y = decoder_layer(z, z)  # Using encoder output as both input and context for testing
print(f"Output shape after encoder+decoder layer: {y.shape}")


Initializing AudioEncoderLayer (d_model=128)
Output shape after encoder+decoder layer: torch.Size([16, 1500, 128])


In [10]:
# And now to decode the results
from mini_whisper.tokenizer.tokenizer import BPE_Tokenizer
from mini_whisper.decoder.textDecoder import TextDecoder
from torch.nn import functional as F

tokenizer = BPE_Tokenizer()
tokenizer.load_merges('mini_whisper/decoder/merges.txt')
seqs = [tokenizer.encode("hello world"),
        tokenizer.encode("this is a test")]          

txt = torch.tensor(seqs, dtype=torch.long)          
B, T = txt.shape                                    
max_len = 1500

# 1) Pad last dim from 3 -> 20 with zeros
pad_len = max_len - T
txt_padded = F.pad(txt, (0, pad_len), value=0)      # [2, 20]

# 2) Expand batch dim from 2 -> 16 by repeating
repeats = (BATCH_SIZE + B - 1) // B               # ceil(16 / 2) = 8
txt_big = txt_padded.repeat(repeats, 1)[:BATCH_SIZE]  # [16, 20]

decoder = TextDecoder(vocab_size=10000+256, d_model=D_MODEL, max_len=1500, n_layers=3, n_heads=4)
with torch.no_grad():
        print(f'x: {txt_big.shape}, z: {z.shape}')  # [16, seq_len, d_model]
        logits = decoder(txt_big, z)
        print(f'logits shape: {logits.shape}')  # Should be [16, seq_len, vocab_size]
        print(F.softmax(logits, dim=-1).shape)  # Should also be [16, seq_len, vocab_size]
        print(F.softmax(logits, dim=-1)[0, 0, :10])  # Print probabilities of first 10 tokens for the first position in the first batch item
        greedy_token = torch.argmax(F.softmax(logits, dim=-1)[0, 0,:])  # [16, seq_len]
        print(f'decoded token for first position in first batch item: {tokenizer.decode([greedy_token.item()])}')
        # Apply softmax over the entire sequence for the first batch item and decode the most probable token at each position
        list(map(lambda i: print(f'greedy token for position {i} in first batch item: {tokenizer.decode([torch.argmax(F.softmax(logits, dim=-1)[0, i, :]).item()])}'), range(20)))
        
        

x: torch.Size([16, 1500]), z: torch.Size([16, 1500, 128])
pos emb shape: torch.Size([16, 1500, 128]), token shape: torch.Size([16, 1500, 128])
logits shape: torch.Size([16, 1500, 10256])
torch.Size([16, 1500, 10256])
tensor([8.6671e-29, 3.3232e-24, 8.8673e-19, 6.0330e-23, 2.5938e-23, 5.6659e-22,
        1.5283e-14, 4.6411e-17, 6.1854e-25, 6.8230e-20])
decoded token for first position in first batch item: kins
greedy token for position 0 in first batch item: kins
greedy token for position 1 in first batch item: kins
greedy token for position 2 in first batch item: kins
greedy token for position 3 in first batch item: kins
greedy token for position 4 in first batch item: kins
greedy token for position 5 in first batch item: kins
greedy token for position 6 in first batch item: kins
greedy token for position 7 in first batch item: kins
greedy token for position 8 in first batch item: kins
greedy token for position 9 in first batch item: kins
greedy token for position 10 in first batch ite

In [11]:
from transformers import WhisperTokenizer
tokenizer = WhisperTokenizer.from_pretrained("openai/whisper-base")
encodedtokenizer = tokenizer.encode("hello world.")
print(f'Encoded tokens for "hello world.": {encodedtokenizer}')
tokenizer.decode(encodedtokenizer)

/mnt/c/Users/red30/Documents/Development/CE8_DeepLearningProject/.venv/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Encoded tokens for "hello world.": [50258, 50363, 675, 1913, 1002, 13, 50257]


'<|startoftranscript|><|notimestamps|>hello world.<|endoftext|>'

In [12]:
seqs = [tokenizer.encode("hello world"),
        tokenizer.encode("this is a test")]   
seqs = [torch.tensor(s, dtype=torch.long) for s in seqs]       

B, T = txt.shape                                    
max_len = 1500

# 1) Pad last dim from 3 -> 20 with zeros
pad_len = max_len - T
txt_padded = F.pad(txt, (0, pad_len), value=0)      # [2, 20]

# 2) Expand batch dim from 2 -> 16 by repeating
repeats = (BATCH_SIZE + B - 1) // B               # ceil(16 / 2) = 8
txt_big = txt_padded.repeat(repeats, 1)[:BATCH_SIZE]  # [16, 20]

decoder = TextDecoder(vocab_size=10000+256, d_model=D_MODEL, max_len=1500, n_layers=3, n_heads=4)
with torch.no_grad():
        print(f'x: {txt_big.shape}, z: {z.shape}')  # [16, seq_len, d_model]
        logits = decoder(txt_big, z)
        print(f'logits shape: {logits.shape}')  # Should be [16, seq_len, vocab_size]
        print(F.softmax(logits, dim=-1).shape)  # Should also be [16, seq_len, vocab_size]
        print(F.softmax(logits, dim=-1)[0, 0, :10])  # Print probabilities of first 10 tokens for the first position in the first batch item
        greedy_token = torch.argmax(F.softmax(logits, dim=-1)[0, 0,:])  # [16, seq_len]
        print(f'decoded token for first position in first batch item: {tokenizer.decode([greedy_token.item()])}')
        # Apply softmax over the entire sequence for the first batch item and decode the most probable token at each position
        list(map(lambda i: print(f'greedy token for position {i} in first batch item: {tokenizer.decode([torch.argmax(F.softmax(logits, dim=-1)[i, 0, :]).item()])}'), range(16)))
        

x: torch.Size([16, 1500]), z: torch.Size([16, 1500, 128])
pos emb shape: torch.Size([16, 1500, 128]), token shape: torch.Size([16, 1500, 128])
logits shape: torch.Size([16, 1500, 10256])
torch.Size([16, 1500, 10256])
tensor([2.1633e-21, 1.6185e-12, 8.8024e-19, 5.0159e-23, 1.9139e-19, 4.3447e-18,
        8.6148e-20, 3.8203e-23, 3.7665e-15, 2.6874e-20])
decoded token for first position in first batch item: iqu
greedy token for position 0 in first batch item: iqu
greedy token for position 1 in first batch item:  AM
greedy token for position 2 in first batch item: iqu
greedy token for position 3 in first batch item:  AM
greedy token for position 4 in first batch item:  AM
greedy token for position 5 in first batch item: iqu
greedy token for position 6 in first batch item: iqu
greedy token for position 7 in first batch item: iqu
greedy token for position 8 in first batch item:  AM
greedy token for position 9 in first batch item: iqu
greedy token for position 10 in first batch item: iqu
gree

In [13]:
from mini_whisper.model import MiniWhisper
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = MiniWhisper(vocab_size=tokenizer.vocab_size+1000, d_model=128).to(DEVICE)

def convert_transcripts_to_targets(transcripts, tokenizer):
    max_len = 448
    BATCH_SIZE = len(transcripts)
    
    seqs = list(map(lambda t: tokenizer.encode(t), transcripts))  # List of lists of token IDs

    # Pad all sequences to the same length (max_len)
    txt_padded = torch.zeros(BATCH_SIZE, max_len, dtype=torch.long)
    for i, s in enumerate(seqs):
        txt_padded[i, :len(s)] = torch.tensor(s[:max_len], dtype=torch.long)

    return txt_padded

In [ ]:
loss_fn = torch.nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=1e-4)
running_loss = 0.0
last_loss = 0.0

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model.to(DEVICE)
loss_fn.to(DEVICE)  # CrossEntropyLoss is stateless but good practice

for i, batch in enumerate(dataloader):
    log_mels = batch['log_mel']  # (B, n_mels, T)
    log_mels = torch.nan_to_num(log_mels, nan=0.0, posinf=10.0, neginf=-10.0)
    log_mels = torch.clamp(log_mels, -10, 10)
    
    if log_mels.shape != (log_mels.shape[0], 80, 3000):
        print(f"Skipping bad batch {i} shape {log_mels.shape}")
        continue
        
    log_mels = log_mels.to(DEVICE)
    
    targets_cpu = convert_transcripts_to_targets(batch['transcript'], tokenizer)
    targets = targets_cpu.to(DEVICE, non_blocking=True)
    
    optimizer.zero_grad()
    outputs = model(log_mels, targets)
    loss = loss_fn(outputs.view(-1, outputs.size(-1)), targets.view(-1))
    
    loss.backward()
    optimizer.step()
        
    
    if i % 10 == 0:
        print(f'Batch {i}, Loss: {loss.item():.4f}')


pos emb shape: torch.Size([16, 448, 128]), token shape: torch.Size([16, 448, 128])
Batch 0, Loss: 60.3074
pos emb shape: torch.Size([16, 448, 128]), token shape: torch.Size([16, 448, 128])
pos emb shape: torch.Size([16, 448, 128]), token shape: torch.Size([16, 448, 128])
pos emb shape: torch.Size([16, 448, 128]), token shape: torch.Size([16, 448, 128])
pos emb shape: torch.Size([16, 448, 128]), token shape: torch.Size([16, 448, 128])
pos emb shape: torch.Size([16, 448, 128]), token shape: torch.Size([16, 448, 128])
pos emb shape: torch.Size([16, 448, 128]), token shape: torch.Size([16, 448, 128])
pos emb shape: torch.Size([16, 448, 128]), token shape: torch.Size([16, 448, 128])
pos emb shape: torch.Size([16, 448, 128]), token shape: torch.Size([16, 448, 128])
pos emb shape: torch.Size([16, 448, 128]), token shape: torch.Size([16, 448, 128])
pos emb shape: torch.Size([16, 448, 128]), token shape: torch.Size([16, 448, 128])
Batch 10, Loss: 5.0050
pos emb shape: torch.Size([16, 448, 128])

In [ ]:
import torch
DEVICE = torch.device('cuda')
t = torch.randn(1, 80, 3000)
t = t.to(DEVICE)
print("Tensor.to(DEVICE) works!")

AcceleratorError: CUDA error: device-side assert triggered
Search for `cudaErrorAssert' in https://docs.nvidia.com/cuda/cuda-runtime-api/group__CUDART__TYPES.html for more information.
CUDA kernel errors might be asynchronously reported at some other API call, so the stacktrace below might be incorrect.
For debugging consider passing CUDA_LAUNCH_BLOCKING=1
Compile with `TORCH_USE_CUDA_DSA` to enable device-side assertions.
